In [1]:
%pip install -qq 'crewai[tools]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.0/559.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 4.9 MB/s eta 0:0

In [3]:
import os
from google.colab import userdata
os.environ['AZURE_API_KEY'] = userdata.get('AZURE_API_KEY')
os.environ['AZURE_OPENAI_API_KEY'] = userdata.get('AZURE_API_KEY')
os.environ['AZURE_API_VERSION'] = userdata.get('AZURE_API_VERSION')
os.environ['AZURE_API_BASE'] = userdata.get('AZURE_BASE_URL')
os.environ['AZURE_OPENAI_ENDPOINT'] = userdata.get('AZURE_BASE_URL')
os.environ['SERPER_API_KEY'] = userdata.get('SERPER_DEV_API_KEY')

In [13]:
from crewai import LLM
from pydantic import BaseModel, Field
class Section(BaseModel):
    answer: str = Field(description="The answer to the question.")
class SuperSection(Section):
    data: Section = Field(description="The answer to the question.")
llm = LLM(f"azure/gpt-4o",
         response_format=Section
    # response_format={"type": "json_object"}
          )
try:
    llm.call(f"What is the capital of France?, response following JSON schema:{SuperSection.model_json_schema()}")
except Exception as e:
    print(e)

 
[2025-03-14 23:30:44][🤖 LLM CALL STARTED]: 2025-03-14 23:30:44.749790


The model azure/gpt-4o does not support response_format for provider 'azure'. Please remove response_format or use 
a supported model.

In [14]:
from crewai import LLM
from pydantic import BaseModel, Field
class Section(BaseModel):
    answer: str = Field(description="The answer to the question.")
class SuperSection(Section):
    data: Section = Field(description="The answer to the question.")
llm = LLM(f"azure/gpt-4o",
        #   , response_format=Section
    # response_format={"type": "json_object"}
          )
llm.call(f"What is the capital of France?, response following JSON schema:{SuperSection.model_json_schema()}")

 
[2025-03-14 23:30:47][🤖 LLM CALL STARTED]: 2025-03-14 23:30:47.720779
 
[2025-03-14 23:30:48][✅ LLM CALL COMPLETED]: 2025-03-14 23:30:48.474585


'```json\n{\n  "answer": "Paris",\n  "data": {\n    "answer": "Paris"\n  }\n}\n```'

In [5]:
SuperSection.model_json_schema()

{'$defs': {'Section': {'properties': {'answer': {'description': 'The answer to the question.',
     'title': 'Answer',
     'type': 'string'}},
   'required': ['answer'],
   'title': 'Section',
   'type': 'object'}},
 'properties': {'answer': {'description': 'The answer to the question.',
   'title': 'Answer',
   'type': 'string'},
  'data': {'$ref': '#/$defs/Section',
   'description': 'The answer to the question.'}},
 'required': ['answer', 'data'],
 'title': 'SuperSection',
 'type': 'object'}

In [15]:
# Example: Research Crew for market analysis
from crewai import Agent, Crew, Process, Task

# Create specialized agents
researcher = Agent(
    llm=llm,
    role="Market Research Specialist",
    goal="Find comprehensive market data on emerging technologies",
    backstory="You are an expert at discovering market trends and gathering data."
)

analyst = Agent(
    llm=llm,
    role="Market Analyst",
    goal="Analyze market data and identify key opportunities",
    backstory="You excel at interpreting market data and spotting valuable insights."
)

# Define their tasks
research_task = Task(
    description="Research the current market landscape for AI-powered healthcare solutions",
    expected_output="Comprehensive market data including key players, market size, and growth trends",
    agent=researcher
)

analysis_task = Task(
    description="Analyze the market data and identify the top 3 investment opportunities",
    expected_output="Analysis report with 3 recommended investment opportunities and rationale",
    agent=analyst,
    context=[research_task]
)

# Create the crew
market_analysis_crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True
)

# Run the crew
result = market_analysis_crew.kickoff()
from IPython import display
display.Markdown(result.raw)

 
[2025-03-14 23:31:25][🚀 CREW 'CREW' STARTED, 47AA4184-1CAF-4949-B818-9F6B55C5ABB2]: 2025-03-14 23:31:25.041646
 
[2025-03-14 23:31:25][📋 TASK STARTED: RESEARCH THE CURRENT MARKET LANDSCAPE FOR AI-POWERED HEALTHCARE SOLUTIONS]: 2025-03-14 23:31:25.056089
 
[2025-03-14 23:31:25][🤖 AGENT 'MARKET RESEARCH SPECIALIST' STARTED TASK]: 2025-03-14 23:31:25.057076
# Agent: Market Research Specialist
## Task: Research the current market landscape for AI-powered healthcare solutions
 
[2025-03-14 23:31:25][🤖 LLM CALL STARTED]: 2025-03-14 23:31:25.057256
 
[2025-03-14 23:31:30][✅ LLM CALL COMPLETED]: 2025-03-14 23:31:30.611015


# Agent: Market Research Specialist
## Final Answer: 
The market for AI-powered healthcare solutions has been experiencing substantial growth, driven by advancements in machine learning algorithms, the increasing volume of healthcare data, and the need for efficient patient care and operational management.

Market Size and Growth:
- As of 2021, the global AI in healthcare

**Analysis Report: Top 3 Investment Opportunities in AI-Powered Healthcare Solutions**

**1. Clinical Applications - Radiology and Imaging Diagnostics**

**Rationale:**
- **Growth Potential:** Radiology and imaging diagnostics represent one of the most promising areas for AI application in healthcare. AI-driven tools can analyze vast amounts of imaging data rapidly and accurately, assisting radiologists in detecting anomalies and diagnosing conditions. Given the high CAGR of 41.7% in the AI healthcare market, investments in clinical applications, particularly radiology, are expected to yield substantial returns.
- **Current Developments:** Companies like IBM Watson Health and Siemens Healthineers are already making significant strides in this field. Their advanced AI platforms are enhancing diagnostic accuracy, reducing patient wait times, and improving overall patient outcomes.
- **Competitive Advantage:** AI's ability to continuously learn and improve from new data positions it as a critical tool in radiology, where accuracy and speed are paramount. Early investment in this sub-sector can position investors advantageously as the technology becomes more widespread.

**2. Remote Monitoring and Telehealth**

**Rationale:**
- **Demand Surge:** The COVID-19 pandemic has significantly accelerated the adoption of telehealth services. AI-powered remote monitoring tools and virtual health assistants are essential in expanding access to care, particularly for chronic condition management and follow-up care.
- **Key Players:** Companies like Philips Healthcare and Medtronic are leading the development of AI-driven telehealth solutions. Their innovations in remote patient monitoring have proven crucial in managing patient care during and post-pandemic.
- **Market Growth:** The integration of AI in telehealth is transforming how healthcare is delivered, making it more efficient and accessible. As telehealth continues to grow beyond the pandemic, investments in this area are likely to experience exponential growth, driven by ongoing demand and technological advancements.

**3. Drug Discovery and Development**

**Rationale:**
- **Efficiency Gains:** AI is revolutionizing the drug discovery process by significantly reducing the time and cost involved. AI models can predict how different compounds will interact with targets, identify potential side effects, and streamline the development pipeline.
- **Industry Impact:** Corporations such as Google Health (DeepMind) and NVIDIA Corporation are at the forefront of leveraging AI for drug discovery. NVIDIA's AI computing platforms, for example, provide the computational power necessary for complex simulations and data analysis in drug development.
- **Financial Opportunity:** As the pressure to rapidly develop treatments for emerging diseases grows, the reliance on AI to enhance R&D efforts will increase. This offers a lucrative investment opportunity, with a market poised for rapid expansion as new AI applications in drug discovery are commercialized.

**Conclusion:**
The market for AI-powered healthcare solutions presents tremendous investment opportunities. By focusing on key growth areas such as radiology and imaging diagnostics, remote monitoring and telehealth, and drug discovery and development, investors can capitalize on the rapid expansion and technological advancements within the healthcare sector. These investments not only promise financial returns but also contribute to significant improvements in healthcare delivery and patient outcomes.

In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
# Example: Customer Support Flow with structured processing
from crewai.flow.flow import Flow, listen, router, start
from pydantic import BaseModel
from typing import List, Dict

# Define structured state
class SupportTicketState(BaseModel):
    ticket_id: str = ""
    customer_name: str = ""
    issue_description: str = ""
    category: str = ""
    priority: str = "medium"
    resolution: str = ""
    satisfaction_score: int = 0


class CustomerSupportFlow(Flow[SupportTicketState]):

    @start()
    def receive_ticket(self):
        # In a real app, this might come from an API
        self.state.ticket_id = "TKT-12345"
        self.state.customer_name = "Alex Johnson"
        self.state.issue_description = "Unable to access premium features after payment"
        return "Ticket received"

    @listen(receive_ticket)
    def categorize_ticket(self, _):
        # Use a direct LLM call for categorization
        from crewai import LLM
        llm = LLM(model="azure/gpt-4o")

        prompt = f"""
        Categorize the following customer support issue into one of these categories:
        - Billing
        - Account Access
        - Technical Issue
        - Feature Request
        - Other

        Issue: {self.state.issue_description}

        Return only the category name.
        """

        self.state.category = llm.call(prompt).strip()
        return self.state.category

    @router(categorize_ticket)
    def route_by_category(self, category):
        # Route to different handlers based on category
        return category.lower().replace(" ", "_")

    @listen("billing")
    def handle_billing_issue(self):
        # Handle billing-specific logic
        self.state.priority = "high"
        # More billing-specific processing...
        return "Billing issue handled"

    @listen("account_access")
    def handle_access_issue(self):
        # Handle access-specific logic
        self.state.priority = "high"
        # More access-specific processing...
        return "Access issue handled"

    # Additional category handlers...

    @listen(handle_billing_issue)
    # @listen("billing")
    def resolve_ticket(self, resolution_info):
        # Final resolution step
        self.state.resolution = f"Issue resolved: {resolution_info}"
        return self.state.resolution

# Run the flow
support_flow = CustomerSupportFlow()
result = support_flow.kickoff()
print(result)

 
[2025-03-14 22:52:34][🌊 FLOW CREATED: 'CUSTOMERSUPPORTFLOW']: 2025-03-14 22:52:34.571010
 
[2025-03-14 22:52:34][🤖 FLOW STARTED: 'CUSTOMERSUPPORTFLOW', 690F7762-DA22-4C60-BFF6-62F05B9A9A35]: 2025-03-14 22:52:34.572743
 Flow started with ID: 690f7762-da22-4c60-bff6-62f05b9a9a35
 
[2025-03-14 22:52:34][🤖 FLOW METHOD STARTED: 'RECEIVE_TICKET']: 2025-03-14 22:52:34.573645
 
[2025-03-14 22:52:34][👍 FLOW METHOD FINISHED: 'RECEIVE_TICKET']: 2025-03-14 22:52:34.574121
 
[2025-03-14 22:52:34][🤖 FLOW METHOD STARTED: 'CATEGORIZE_TICKET']: 2025-03-14 22:52:34.574434
 
[2025-03-14 22:52:34][🤖 LLM CALL STARTED]: 2025-03-14 22:52:34.574662
 
[2025-03-14 22:52:34][✅ LLM CALL COMPLETED]: 2025-03-14 22:52:34.852007
 
[2025-03-14 22:52:34][👍 FLOW METHOD FINISHED: 'CATEGORIZE_TICKET']: 2025-03-14 22:52:34.852731
 
[2025-03-14 22:52:34][🤖 FLOW METHOD STARTED: 'ROUTE_BY_CATEGORY']: 2025-03-14 22:52:34.852900
 
[2025-03-14 22:52:34][👍 FLOW METHOD FINISHED: 'ROUTE_BY_CATEGORY']: 2025-03-14 22:52:34.852981
 

In [10]:
# Example: Content Production Pipeline combining Crews and Flows
from crewai.flow.flow import Flow, listen, start
from crewai import Agent, Crew, Process, Task
from pydantic import BaseModel
from typing import List, Dict
from rich import print
class ContentState(BaseModel):
    topic: str = ""
    target_audience: str = ""
    content_type: str = ""
    outline: Dict = {}
    draft_content: str = ""
    final_content: str = ""
    seo_score: int = 0

class ContentProductionFlow(Flow[ContentState]):
    @start()
    def initialize_project(self):
        # Set initial parameters
        self.state.topic = "Sustainable Investing"
        self.state.target_audience = "Millennial Investors"
        self.state.content_type = "Blog Post"
        return "Project initialized"

    @listen(initialize_project)
    def create_outline(self, _):
        # Use a research crew to create an outline
        researcher = Agent(
            llm=llm,
            role="Content Researcher",
            goal=f"Research {self.state.topic} for {self.state.target_audience}",
            backstory="You are an expert researcher with deep knowledge of content creation."
        )

        outliner = Agent(
            llm=llm,
            role="Content Strategist",
            goal=f"Create an engaging outline for a {self.state.content_type}",
            backstory="You excel at structuring content for maximum engagement."
        )

        research_task = Task(
            description=f"Research {self.state.topic} focusing on what would interest {self.state.target_audience}",
            expected_output="Comprehensive research notes with key points and statistics",
            agent=researcher
        )

        outline_task = Task(
            description=f"Create an outline for a {self.state.content_type} about {self.state.topic}",
            expected_output="Detailed content outline with sections and key points",
            agent=outliner,
            context=[research_task]
        )

        outline_crew = Crew(
            agents=[researcher, outliner],
            tasks=[research_task, outline_task],
            process=Process.sequential,
            verbose=True
        )

        # Run the crew and store the result
        result = outline_crew.kickoff()

        # Parse the outline (in a real app, you might use a more robust parsing approach)
        import json, re
        try:
            str_result = str(result)
            if str_result.startswith("```"):
                content_json = re.sub(r"```(?:\w+\n)?(.*?)```", r"\1", str_result, flags=re.DOTALL).strip()
            self.state.outline = json.loads(content_json)
        except:
            # Fallback if not valid JSON
            self.state.outline = {"sections": result.raw}

        return "Outline created"

    @listen(create_outline)
    def write_content(self, _):
        # Use a writing crew to create the content
        writer = Agent(
            llm=llm,
            role="Content Writer",
            goal=f"Write engaging content for {self.state.target_audience}",
            backstory="You are a skilled writer who creates compelling content."
        )

        editor = Agent(
            llm=llm,
            role="Content Editor",
            goal="Ensure content is polished, accurate, and engaging",
            backstory="You have a keen eye for detail and a talent for improving content."
        )

        writing_task = Task(
            description=f"Write a {self.state.content_type} about {self.state.topic} following this outline: {self.state.outline}",
            expected_output="Complete draft content in markdown format",
            agent=writer
        )

        editing_task = Task(
            description="Edit and improve the draft content for clarity, engagement, and accuracy",
            expected_output="Polished final content in markdown format",
            agent=editor,
            context=[writing_task]
        )

        writing_crew = Crew(
            agents=[writer, editor],
            tasks=[writing_task, editing_task],
            process=Process.sequential,
            verbose=True
        )

        # Run the crew and store the result
        result = writing_crew.kickoff()
        self.state.final_content = result.raw

        return "Content created"

    @listen(write_content)
    def optimize_for_seo(self, _):
        # Use a direct LLM call for SEO optimization
        from crewai import LLM
        llm = LLM(model="azure/gpt-4o")

        prompt = f"""
        Analyze this content for SEO effectiveness for the keyword "{self.state.topic}".
        Rate it on a scale of 1-100 and provide 3 specific recommendations for improvement.

        Content: {self.state.final_content[:1000]}... (truncated for brevity)

        Format your response as JSON with the following structure:
        {{
            "score": 85,
            "recommendations": [
                "Recommendation 1",
                "Recommendation 2",
                "Recommendation 3"
            ]
        }}
        """

        seo_analysis = llm.call(prompt)

        # Parse the SEO analysis
        import json, re
        try:
            seo_analysis = re.sub(r"```(?:\w+\n)?(.*?)```", r"\1", seo_analysis, flags=re.DOTALL).strip()
            analysis = json.loads(seo_analysis)
            self.state.seo_score = analysis.get("score", 0)
            return analysis
        except:
            self.state.seo_score = 50
            return {"score": 50, "recommendations": ["Unable to parse SEO analysis"]}

# Run the flow
content_flow = ContentProductionFlow()
result = content_flow.kickoff()
print(result)

 
[2025-03-14 22:54:23][🌊 FLOW CREATED: 'CONTENTPRODUCTIONFLOW']: 2025-03-14 22:54:23.365226
 
[2025-03-14 22:54:23][🤖 FLOW STARTED: 'CONTENTPRODUCTIONFLOW', 583CBE2D-07A7-48A1-BD0B-DF4843443C00]: 2025-03-14 22:54:23.366024
 Flow started with ID: 583cbe2d-07a7-48a1-bd0b-df4843443c00
 
[2025-03-14 22:54:23][🤖 FLOW METHOD STARTED: 'INITIALIZE_PROJECT']: 2025-03-14 22:54:23.366634
 
[2025-03-14 22:54:23][👍 FLOW METHOD FINISHED: 'INITIALIZE_PROJECT']: 2025-03-14 22:54:23.367167
 
[2025-03-14 22:54:23][🤖 FLOW METHOD STARTED: 'CREATE_OUTLINE']: 2025-03-14 22:54:23.367475
 
[2025-03-14 22:54:23][🚀 CREW 'CREW' STARTED, F05731EC-1454-49AC-A1F4-B7BE530A355E]: 2025-03-14 22:54:23.373187
 
[2025-03-14 22:54:23][📋 TASK STARTED: RESEARCH SUSTAINABLE INVESTING FOCUSING ON WHAT WOULD INTEREST MILLENNIAL INVESTORS]: 2025-03-14 22:54:23.386857
 
[2025-03-14 22:54:23][🤖 AGENT 'CONTENT RESEARCHER' STARTED TASK]: 2025-03-14 22:54:23.387920
# Agent: Content Researcher
## Task: Research Sustainable Investing

{
    'score': 75,
    'recommendations': [
        "Increase keyword density: While the content is informative, the keyword 'Sustainable Investing' should be 
incorporated more frequently and naturally throughout the content to improve relevance and SEO ranking.",
        'Add internal and external links: Include links to other relevant blog posts on your site (internal links) 
and reputable sources or studies related to sustainable investing (external links) to enhance credibility and SEO 
strength.',
        'Include multimedia elements: Enhance the content with images, charts, or videos related to sustainable 
investing to make it more engaging and improve the chances of ranking higher in search results due to rich media 
content.'
    ]
}